In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [2]:
class FeedforwardNet(nn.Module):
    def __init__(self, input_size=12288, hidden_size_1=256, hidden_size_2=128, hidden_size_3=64, speed_size=1,
                 output_size=1):
        super(FeedforwardNet, self).__init__()
        self.device = 'cuda'
        self.fc1 = nn.Linear(input_size, hidden_size_1).to(self.device)
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2).to(self.device)
        self.fc3 = nn.Linear(hidden_size_2, hidden_size_3).to(self.device)
        self.fc4 = nn.Linear(hidden_size_3 + speed_size, output_size).to(self.device)
        self.relu = nn.ReLU().to(self.device)
        self.tanh = nn.Tanh().to(self.device)

    def forward(self, image, speed):
        image = image.to(self.device)
        speed = speed.to(self.device)
        out = self.fc1(image).to(self.device)
        out = self.relu(out).to(self.device)
        out = self.fc2(out).to(self.device)
        out = self.relu(out).to(self.device)
        out = self.fc3(out).to(self.device)
        out = self.relu(out).to(self.device)
        
        out = torch.cat((out, speed), dim=1).to(self.device)

        out = self.fc4(out).to(self.device)
        output = self.tanh(out).to(self.device)
        return output

In [3]:
# Подготовка данных (пример)
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_wheel.csv')

# Ensure the first dimension (batch size) is the same for all tensors
batch_size = min(len(X), len(S), len(y))
X_tensor = torch.tensor(X.values[:batch_size], dtype=torch.float32).to(device='cuda')
S_tensor = torch.tensor(S.values[:batch_size], dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values[:batch_size, 0], dtype=torch.float32).unsqueeze(1).to(device='cuda')
# Create the dataset
dataset = TensorDataset(X_tensor, S_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

In [4]:
print(len(X_tensor))
print(len(S_tensor))
print(len(y_tensor))

13118
13118
13118


In [5]:
print(X_tensor[0])
print(type(X_tensor[0]))
print(X_tensor[0].size())

tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0')
<class 'torch.Tensor'>
torch.Size([12288])


In [17]:
print(S_tensor[0])
print(type(S_tensor[0]))
print(S_tensor[0].size())

tensor([104.], device='cuda:0')

In [18]:
print(y_tensor[0])
print(type(y_tensor[0]))
print(y_tensor[0].size())

tensor([0.0296], device='cuda:0')

In [19]:
128 * 96

12288

In [21]:
# Создание экземпляра модели
model = FeedforwardNet().to(device='cuda')

# Определение функции потерь и оптимизатора
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Обучение модели
num_epochs = 200
for epoch in range(num_epochs):
    for X, S, y in train_loader:
        # Прямое прохождение
        output = model(X, S)
        loss = criterion(output, y)
        
        # Обратное распространение и обновление весов
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.9f}')

Epoch [1/200], Loss: 0.799605370
Epoch [2/200], Loss: 0.701763630
Epoch [3/200], Loss: 0.611487627
Epoch [4/200], Loss: 0.632477403
Epoch [5/200], Loss: 0.618355870
Epoch [6/200], Loss: 0.579983711
Epoch [7/200], Loss: 0.538627088
Epoch [8/200], Loss: 0.517619014
Epoch [9/200], Loss: 0.497562021
Epoch [10/200], Loss: 0.476423174
Epoch [11/200], Loss: 0.469636023
Epoch [12/200], Loss: 0.453543574
Epoch [13/200], Loss: 0.433928967
Epoch [14/200], Loss: 0.421092957
Epoch [15/200], Loss: 0.403477311
Epoch [16/200], Loss: 0.385730356
Epoch [17/200], Loss: 0.372589201
Epoch [18/200], Loss: 0.355060637
Epoch [19/200], Loss: 0.336741567
Epoch [20/200], Loss: 0.320858836
Epoch [21/200], Loss: 0.302477449
Epoch [22/200], Loss: 0.284422517
Epoch [23/200], Loss: 0.267039359
Epoch [24/200], Loss: 0.246095181
Epoch [25/200], Loss: 0.225563198
Epoch [26/200], Loss: 0.203556493
Epoch [27/200], Loss: 0.179882362
Epoch [28/200], Loss: 0.158599198
Epoch [29/200], Loss: 0.136371776
Epoch [30/200], Loss: 0

In [22]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn.pth')

In [23]:
print(X_tensor.size(0))
print(S_tensor.size(0))
print(y_tensor.size(0))

13118
13118
13118
